# Volatility models on total return (Michal)

Stream 2, volatility and risk. Owner of this notebook: Michal. Olivia's `analysis/volatility.py` is left as it is; this notebook redoes the volatility work on total return and with Kritzman's recommendations of 2026-10-13.

Plan:
1. Set up and fetch the data.
2. **Choose the total return series**: compare the candidates before using one.
3. Next: trailing volatility, EWMA and historical simulation, with data up to the day before each month-end.
4. Later: GARCH, t-GARCH, GJR-GARCH.

AI use: drafted with Claude Code; the session file says what it contributed.

## 1. Setup

Installs what the notebook needs into the running Python. `xlrd` reads Shiller's `.xls` file.

In [ ]:
%pip install -q yfinance xlrd arch

In [ ]:
import os

# One core for the numerical libraries, so a long run never takes over the computer.
for var in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"]:
    os.environ.setdefault(var, "1")

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

# The repository root, whether the notebook runs from analysis/ or from the root.
ROOT = Path.cwd()
while not (ROOT / "README.md").exists():
    ROOT = ROOT.parent
LICENSED = ROOT / "data" / "licensed"  # gitignored: Yahoo's terms forbid redistributing the prices
LICENSED.mkdir(parents=True, exist_ok=True)
print(ROOT.name)

## 2. Fetch the S&P 500 from Yahoo Finance

- `^GSPC`: the price index, daily from 1927. Saved as `data/licensed/sp500_yahoo_raw.csv` in the same layout as Samantha's copy (dates `M/D/YYYY`), so `analysis/baseline.py` and `analysis/volatility.py` read it unchanged.
- `^SP500TR`: S&P's own total return index, dividends reinvested, daily from 1988. The benchmark for checking any longer total return series we build.

Both stay in `data/licensed/`, which git ignores.

In [ ]:
def fetch(ticker):
    raw = yf.download(ticker, period="max", interval="1d", auto_adjust=False, progress=False)
    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = raw.columns.get_level_values(0)
    raw = raw.dropna(subset=["Close"])
    raw.index = pd.to_datetime(raw.index).tz_localize(None)
    return raw


gspc = fetch("^GSPC")
sptr = fetch("^SP500TR")

for df, name in [(gspc, "sp500_yahoo_raw.csv"), (sptr, "sp500tr_yahoo_raw.csv")]:
    out = df[["Open", "High", "Low", "Close", "Volume"]].copy()
    out["Dividends"] = 0.0
    out["Stock Splits"] = 0.0
    out.index = [f"{d.month}/{d.day}/{d.year}" for d in out.index]
    out.index.name = "Date"
    out.to_csv(LICENSED / name)

for name, df in [("^GSPC", gspc), ("^SP500TR", sptr)]:
    print(f"{name}: {df.index.min():%Y-%m-%d} to {df.index.max():%Y-%m-%d}, {len(df):,} days")

## 3. Shiller's monthly dividends

`D` in Shiller's file is the dividend per share of the S&P composite **at an annual rate**, in index points (the same units as the price `P`). Shiller builds the monthly values by interpolating quarterly figures, so his own monthly series is already smoothed. Recent months are estimates and get revised.

In [ ]:
xls = pd.read_excel(ROOT / "data/valuation/ie_data_raw.xls", sheet_name="Data", header=None)
header_row = xls.index[xls.iloc[:, 0].astype(str).str.strip() == "Date"][0]
shiller = xls.iloc[header_row + 1:, :3].copy()
shiller.columns = ["Date", "P", "D"]
shiller = shiller.apply(pd.to_numeric, errors="coerce").dropna(subset=["Date", "P"])

# Shiller writes October as 1871.1, so the month is the first two decimals.
year = shiller["Date"].astype(int)
month = ((shiller["Date"] - year) * 100).round().astype(int)
shiller.index = pd.PeriodIndex.from_fields(year=year, month=month, freq="M")
shiller = shiller[["P", "D"]]
print(shiller.dropna().tail(3))
print("Dividends available to", shiller["D"].dropna().index.max())

## 4. Candidate total return series

Price return plus the dividend received that day: `R_TR(t) = (P(t) + div(t)) / P(t-1) - 1`. The only question is where each month's dividend, `D/12` index points, falls within the month:

- **A. Spread evenly** over the month's trading days.
- **B. Paid in one lump** on the month's last trading day. Deliberately crude: if A and B give nearly the same 3-month returns, where the dividend falls inside a month does not matter for us.

Other candidates, not built here:
- **C.** S&P's own `^SP500TR`: exact, but only from 1988. Used below as the benchmark.
- **D.** CRSP `vwretd` from the WRDS S&P 500 file: exact back to 1926, but licensed, and the team's downloads so far start in 1962.
- **E.** The French market return `Mkt-RF + RF`: exact dividends from 1926, but the whole US market, not the S&P 500. Compared in section 6.

In [ ]:
price = gspc["Close"]
month = price.index.to_period("M")
monthly_div_points = shiller["D"].reindex(month).to_numpy() / 12  # index points paid in that month
days_in_month = price.groupby(month).transform("size").to_numpy()
is_last_day = ~month.duplicated(keep="last")

div_even = pd.Series(monthly_div_points / days_in_month, index=price.index)
div_lump = pd.Series(np.where(is_last_day, monthly_div_points, 0.0), index=price.index)

daily = pd.DataFrame({
    "price": price.pct_change(),
    "tr_even": (price + div_even) / price.shift(1) - 1,
    "tr_lump": (price + div_lump) / price.shift(1) - 1,
    "tr_sp500tr": sptr["Close"].pct_change(),
}).dropna(subset=["price", "tr_even"])
print(f"Built {daily.index.min():%Y-%m-%d} to {daily.index.max():%Y-%m-%d}; "
      f"Shiller dividends end {shiller['D'].dropna().index.max()}")

## 5. Check against S&P's own total return index (1988 on)

What matters for the project:
- **3-month returns** at month-ends, since the target is a 10% loss over three months: how far the built series is from S&P's, and whether the 10% events agree.
- **Daily volatility**, for the volatility models: do the dividends change it at all?
- **The annual return**, where the dividends show up (Kritzman: total return, a dividend yield of about 3.5–4% historically).

In [ ]:
common = daily.dropna(subset=["tr_sp500tr"])
levels = (1 + common).cumprod()
month_end = levels.groupby(levels.index.to_period("M")).tail(1)
ret3 = month_end / month_end.shift(3) - 1
benchmark_event = ret3["tr_sp500tr"] <= -0.10

rows = {}
for col in ["price", "tr_even", "tr_lump", "tr_sp500tr"]:
    gap = (ret3[col] - ret3["tr_sp500tr"]).dropna()
    event = ret3[col] <= -0.10
    rows[col] = {
        "3m gap vs S&P TR, mean": gap.mean(),
        "3m gap, std": gap.std(),
        "3m gap, max abs": gap.abs().max(),
        "10% events": int(event.sum()),
        "events differing from S&P TR": int((event != benchmark_event)[gap.index].sum()),
        "daily vol, annualized": common[col].std() * np.sqrt(252),
        "annual return": levels[col].iloc[-1] ** (252 / len(common)) - 1,
    }
summary = pd.DataFrame(rows).T
print(f"{common.index.min():%Y-%m-%d} to {common.index.max():%Y-%m-%d}")
summary.round(4)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
(levels["tr_even"] / levels["tr_sp500tr"]).plot(ax=ax[0], label="A: spread evenly")
(levels["tr_lump"] / levels["tr_sp500tr"]).plot(ax=ax[0], label="B: lump at month-end")
(levels["price"] / levels["tr_sp500tr"]).plot(ax=ax[0], label="price only", ls="--")
ax[0].axhline(1, color="grey", lw=0.8)
ax[0].set_title("Built series / S&P 500 TR index")
ax[0].legend()
(ret3["tr_even"] - ret3["tr_sp500tr"]).plot(ax=ax[1])
ax[1].set_title("3-month return gap, A minus S&P TR")
plt.tight_layout()

## 6. Check against the French market return (1926 on)

A different index (the whole US market) with exact dividends. It cannot confirm the S&P level, but it checks the built series before 1988, where S&P's own index does not exist.

In [ ]:
ff = pd.read_csv(ROOT / "data/equity-factors/F-F_Research_Data_Factors_daily_raw.csv", skiprows=4)
ff = ff.rename(columns={ff.columns[0]: "Date"})
ff = ff[ff["Date"].astype(str).str.strip().str.fullmatch(r"\d{8}")]
ff.index = pd.to_datetime(ff["Date"].astype(str).str.strip(), format="%Y%m%d")
french = ((ff["Mkt-RF"].astype(float) + ff["RF"].astype(float)) / 100).rename("french")

both = pd.concat([daily["tr_even"], french], axis=1, join="inner").dropna()
lv = (1 + both).cumprod()
me = lv.groupby(lv.index.to_period("M")).tail(1)
r3 = (me / me.shift(3) - 1).dropna()
for start, end in [("1927", "1987"), ("1988", "2026")]:
    s = r3.loc[start:end]
    agree = ((s["tr_even"] <= -0.10) == (s["french"] <= -0.10)).mean()
    print(f"{start}-{end}: 3m return correlation {s.corr().iloc[0, 1]:.3f}; "
          f"10% events: built S&P {int((s['tr_even'] <= -0.10).sum())}, French {int((s['french'] <= -0.10).sum())}; "
          f"months agreeing {agree:.1%}")

## 7. Decision

Where the dividend falls inside a month makes no difference: A and B both track S&P's own index to within a fraction of a percent over three months and give the same 23 loss events since 1988. So the total return series is **joined**: A (Yahoo price plus Shiller's dividends spread evenly) from 1928 to 1987, and S&P's own `^SP500TR` from January 1988, chained by daily returns. The join also runs to today, while Shiller's dividends stop a few months back. To agree with Samantha, since the event series and the baseline must switch to the same series.

## 8. The joined series

Checks on the joined series:
1. **The seam**: 3-month returns of windows that straddle January 1988, joined against built-only (A throughout).
2. **Joined against built-only over the whole overlap**: does the choice of series change any 10% event?
3. **What the dividends add**, decade by decade: total return minus price return, against Kritzman's "about 3.5–4% historically, much lower now".
4. **The event series** on total return against the price-only events the team uses now.

In [ ]:
# S&P's index has no return on its first day (1988-01-04), so the built series covers that day.
sp_ret = sptr["Close"].pct_change().dropna()
JOIN = sp_ret.index[0]
tr_daily = pd.concat([
    daily.loc[daily.index < JOIN, "tr_even"],
    sp_ret,
]).rename("tr")
# Keep complete months only: the current month has no month-end yet.
LAST_COMPLETE = pd.Timestamp.today().to_period("M") - 1
tr_daily = tr_daily[tr_daily.index.to_period("M") <= LAST_COMPLETE]
print(f"Joined: built A to {daily.index[daily.index < JOIN].max():%Y-%m-%d}, S&P TR from {JOIN:%Y-%m-%d} "
      f"to {tr_daily.index.max():%Y-%m-%d}")
tr_level = (1 + tr_daily).cumprod().rename("tr_level")

price_daily = gspc["Close"].pct_change().reindex(tr_daily.index)

def month_end_3m(level):
    me = level.groupby(level.index.to_period("M")).tail(1)
    me.index = me.index.to_period("M")
    return (me.shift(-3) / me - 1)  # forward 3-month return from each month-end

fwd_joined = month_end_3m(tr_level)
fwd_built = month_end_3m((1 + daily["tr_even"]).cumprod())
fwd_price = month_end_3m((1 + price_daily).cumprod())

# 1. The seam: windows starting from 1987-07 to 1988-03 straddle or follow the join
seam = pd.DataFrame({"joined": fwd_joined, "built A": fwd_built}).loc["1987-06":"1988-03"]
seam["gap"] = seam["joined"] - seam["built A"]
print("Forward 3-month returns around the join:")
print(seam.round(4).to_string())

# 2. Joined vs built-only where both exist
both = pd.DataFrame({"joined": fwd_joined, "built": fwd_built}).dropna()
print(f"\nJoined vs built-only, {both.index.min()} to {both.index.max()}: "
      f"max abs 3m gap {(both['joined'] - both['built']).abs().max():.4f}, "
      f"10% events joined {int((both['joined'] <= -0.10).sum())}, built {int((both['built'] <= -0.10).sum())}, "
      f"months that differ {int(((both['joined'] <= -0.10) != (both['built'] <= -0.10)).sum())}")

In [ ]:
# 3. Dividend contribution by decade: annualized total return minus annualized price return
yearly = pd.DataFrame({"tr": tr_daily, "price": price_daily}).dropna()
decade = (yearly.index.year // 10) * 10
by_decade = yearly.groupby(decade).apply(
    lambda g: pd.Series({
        "total return": (1 + g["tr"]).prod() ** (252 / len(g)) - 1,
        "price return": (1 + g["price"]).prod() ** (252 / len(g)) - 1,
    })
)
by_decade["dividends add"] = by_decade["total return"] - by_decade["price return"]
by_decade.index = [f"{d}s" for d in by_decade.index]
by_decade.round(4)

In [ ]:
# 4. Event series: total return against price only, every month-end with a full 3-month window
events = pd.DataFrame({"tr": fwd_joined, "price": fwd_price}).dropna()
events["event_tr"] = events["tr"] <= -0.10
events["event_price"] = events["price"] <= -0.10
print(f"{events.index.min()} to {events.index.max()}, {len(events)} month-ends")
print(f"10% loss events: total return {events['event_tr'].sum()}, price only {events['event_price'].sum()}")
for start in ["1928", "1976"]:
    e = events.loc[start:]
    print(f"Base rate from {start}: total return {e['event_tr'].mean():.1%}, price only {e['event_price'].mean():.1%}")
dropped = events.index[events["event_price"] & ~events["event_tr"]]
print("\nMonth-ends that are events on price but not on total return:")
print(", ".join(str(p) for p in dropped))

## 9. Trailing volatility and EWMA on total return, with Kritzman's timing

Changes from `analysis/volatility.py`:
- **Total return**: daily log returns of the joined series.
- **Timing**: each month-end forecast uses data up to the **trading day before the last trading day of the month** (Kritzman, 2026-10-13: you cannot trade at a close that has already happened). The outcome still runs from the month-end close.

The rest follows Olivia's method:
- **Trailing volatility**: standard deviation of the last 63 daily log returns, annualized (Kritzman's "trailing volatility" predictor).
- **EWMA**: RiskMetrics, λ = 0.94.
- **Mean**: the expanding average of daily log returns, annualized, at least a year of data, so nothing from the future.
- **Probability of a 10% loss at the end of the quarter**: `Φ((ln 0.9 − m·T) / (σ·√T))`, T = 0.25.

On the mean (Kritzman's arithmetic-mean point): the formula is for continuous (log) returns, so its drift `m` is the mean *log* return, which equals the arithmetic mean minus σ²/2. Using the mean of daily log returns is therefore the consistent choice; plugging in an arithmetic mean here would count the volatility term twice.

In [ ]:
from scipy.stats import norm
from sklearn.metrics import roc_auc_score

log_tr = np.log1p(tr_daily)

trailing = log_tr.rolling(63).std() * np.sqrt(252)
ewma = np.sqrt(log_tr.pow(2).ewm(alpha=1 - 0.94, adjust=False).mean() * 252)
mu = log_tr.expanding(min_periods=252).mean() * 252

# The forecast date's information: the second-to-last trading day of each month
period = log_tr.index.to_period("M")
day_before_last = log_tr.groupby(period).apply(lambda s: s.index[-2] if len(s) > 1 else s.index[-1])

def at_day_before(series):
    out = series.reindex(day_before_last.values)
    out.index = day_before_last.index
    return out

monthly = pd.DataFrame({
    "vol_trailing63": at_day_before(trailing),
    "vol_ewma94": at_day_before(ewma),
    "mu_expanding": at_day_before(mu),
})
T = 0.25
for name in ["trailing63", "ewma94"]:
    sigma = monthly[f"vol_{name}"]
    monthly[f"p_{name}"] = norm.cdf((np.log(0.9) - monthly["mu_expanding"] * T) / (sigma * np.sqrt(T)))

monthly["ret_3m_tr"] = fwd_joined
monthly["event_tr"] = (monthly["ret_3m_tr"] <= -0.10).astype(float).where(monthly["ret_3m_tr"].notna())
monthly.tail(3).round(4)

### How the probabilities do, before any calibration

Scored on month-ends with a full 3-month outcome, from 1929 (once the mean has a year of data). Brier skill is against the expanding base rate of past events, so it also uses only past data; above zero beats it. AUC: the chance a month with a loss got a higher probability than a month without (0.5 = coin flip).

In [ ]:
scored = monthly.dropna(subset=["event_tr", "p_trailing63", "p_ewma94"]).copy()
# Past base rate: events whose 3-month window had finished by the forecast date
scored["p_base"] = scored["event_tr"].shift(3).expanding(min_periods=12).mean()
scored = scored.dropna(subset=["p_base"])

def brier(p, y):
    return ((p - y) ** 2).mean()

rows = {}
for col in ["p_base", "p_trailing63", "p_ewma94"]:
    rows[col] = {
        "mean p before a loss": scored.loc[scored["event_tr"] == 1, col].mean(),
        "mean p otherwise": scored.loc[scored["event_tr"] == 0, col].mean(),
        "Brier": brier(scored[col], scored["event_tr"]),
        "Brier skill vs base rate": 1 - brier(scored[col], scored["event_tr"]) / brier(scored["p_base"], scored["event_tr"]),
        "AUC": roc_auc_score(scored["event_tr"], scored[col]),
    }
print(f"{scored.index.min()} to {scored.index.max()}: {len(scored)} month-ends, "
      f"{int(scored['event_tr'].sum())} with a 10% loss ({scored['event_tr'].mean():.1%})")
pd.DataFrame(rows).T.round(4)

### What changed against Olivia's series

Her `data/volatility/vol_probabilities.csv` uses the price index and the month-end close. The comparison shows how much each of the two changes matters.

In [ ]:
olivia = pd.read_csv(ROOT / "data/volatility/vol_probabilities.csv", parse_dates=["Date"])
olivia.index = olivia["Date"].dt.to_period("M")
cmp = pd.DataFrame({
    "vol (Olivia)": olivia["vol_rolling63"],
    "vol (Michal)": monthly["vol_trailing63"],
    "p (Olivia)": olivia["p_rolling63"],
    "p (Michal)": monthly["p_trailing63"],
    "mu (Olivia)": olivia["mu_expanding"],
    "mu (Michal)": monthly["mu_expanding"],
}).dropna()
print(f"{cmp.index.min()} to {cmp.index.max()}")
print(f"Trailing vol: correlation {cmp['vol (Olivia)'].corr(cmp['vol (Michal)']):.4f}, "
      f"mean abs difference {(cmp['vol (Olivia)'] - cmp['vol (Michal)']).abs().mean():.4f}")
print(f"Expanding mean: Olivia {cmp['mu (Olivia)'].iloc[-1]:.2%}, Michal {cmp['mu (Michal)'].iloc[-1]:.2%} at the last month")
print(f"Probability: mean Olivia {cmp['p (Olivia)'].mean():.2%}, Michal {cmp['p (Michal)'].mean():.2%}")

fig, ax = plt.subplots(figsize=(12, 3.5))
cmp[["p (Olivia)", "p (Michal)"]].loc["1990":].plot(ax=ax, lw=0.9)
ax.set_title("Probability of a 10% quarterly loss, trailing 63-day volatility")
plt.tight_layout()

## 10. The GARCH family: setup

Each model changes one thing from the one before:

| Model | Variance equation | Shocks | Adds |
|---|---|---|---|
| GARCH(1,1) | σ²ₜ = ω + α·ε²ₜ₋₁ + β·σ²ₜ₋₁ | Normal | Volatility clustering, reversion to a long-run level |
| t-GARCH | same | Student-t, ν estimated | Fat tails |
| GJR-GARCH-t | + γ·ε²ₜ₋₁·[εₜ₋₁ < 0] | Student-t | Leverage: falls raise volatility more than rises |
| GJR-GARCH-skew-t | same as GJR | Hansen's skewed t (η, λ) | Asymmetric tails: about −0.6 skew is left after GJR-t |
| GJR, filtered historical simulation (FHS) | the GJR-t fit | Drawn from the window's own standardised residuals | No distribution assumed for the shocks |

**The mean.** Fitting μ together with the variance gives a mean that is too high: maximum likelihood down-weights the high-volatility days, which are mostly falls (a first run gave about 16% a year on the whole sample, against a mean log return near 10%). Following Danielsson (chapter 2-a, slide 55, "subtract the mean prior to estimation... usually best for risk applications"), each window's returns are **de-meaned** by their sample mean, the variance model is fitted to the de-meaned returns, and that mean is added back wherever returns are forecast.

**No look-ahead, as if run in real time.** At every month-end the models are refitted on a **rolling 12-year window** (about 3,000 trading days, enough for the t's ν; Danielsson asks for several thousand) ending on the trading day before the month's last trading day. Before 1940 the window is shorter, because the data start in 1928. Everything that comes out, the mean included, uses only that window.

**What each month records**, per model:
- **Two probabilities of a 10% loss at the end of the quarter.** *Closed form*: the 63-day variance forecast, added up day by day, in `Φ((ln 0.9 − 63·μ) / σ_Q)`. *Simulation*: 10,000 paths of 64 days drawn from the fitted model, volatility updating along each path; the share of paths that end 10% or more down *is* the probability. The same paths give the share that touch −10% at any point and the 5% quantile of the 3-month return (a 3-month 95% VaR).
- **The parameters, their robust t-statistics, the log-likelihood, AIC and BIC**, so the parameters can be used as data and tested.
- For GJR: **the last 21 days of standardised residuals** (smallest, sum, count beyond ±2.5), the surprises the model has just seen.
- **Daily VaR at 1% and 5%** for every trading day until the next refit, filtered through the actual returns with that month's parameters, for the VaR back-tests.

**Is this VaR?** The loss probability is not VaR: VaR fixes a probability (5%) and asks for the loss; we fix the loss (10%) and ask for the probability. They are two readings of the same simulated distribution, so both are recorded, and the VaR forecasts are back-tested in section 16.

**Running it safely.** The full run takes about seven minutes on one core. Its results are saved to `data/volatility/garch_monthly_michal.csv` and `garch_daily_var_michal.csv` and **read back on later runs instead of being recomputed**. Set the environment variable `GARCH_FORCE=1` to recompute, or `GARCH_MODE=test` for a quick check on 24 months with 200 paths each.

In [ ]:
import time
import warnings

from arch import arch_model
from scipy import stats
from statsmodels.stats.diagnostic import acorr_ljungbox

GARCH_MODE = os.environ.get("GARCH_MODE", "full")
GARCH_FORCE = os.environ.get("GARCH_FORCE", "0") == "1"
WINDOW_DAYS = 12 * 252
FIRST_FORECAST = pd.Period("1938-01", "M")
HORIZON = 64  # day 1 = the month's last trading day; days 2 to 64 = the 63-day outcome window
N_SIMS = 10_000 if GARCH_MODE == "full" else 200
TEST_MONTHS = 24
VAR_LEVELS = [0.01, 0.05]
RESULTS_FILE = ROOT / "data" / "volatility" / "garch_monthly_michal.csv"
DAILY_VAR_FILE = ROOT / "data" / "volatility" / "garch_daily_var_michal.csv"

MODELS = {
    "garch": dict(vol="GARCH", p=1, o=0, q=1, dist="normal"),
    "tgarch": dict(vol="GARCH", p=1, o=0, q=1, dist="t"),
    "gjr": dict(vol="GARCH", p=1, o=1, q=1, dist="t"),
    "gjr_skewt": dict(vol="GARCH", p=1, o=1, q=1, dist="skewt"),
}
SIM_VARIANTS = list(MODELS) + ["gjr_fhs"]  # FHS reuses the GJR-t fit
COLORS = {"garch": "tab:blue", "tgarch": "tab:orange", "gjr": "tab:green", "gjr_skewt": "tab:red", "gjr_fhs": "tab:purple"}
y = (log_tr * 100).rename("r")  # percent: the optimiser is more stable on this scale
print(f"Mode: {GARCH_MODE}, {N_SIMS} paths, window {WINDOW_DAYS} days; data {y.index.min():%Y-%m-%d} to {y.index.max():%Y-%m-%d}")

## 11. Whole-sample fits: parameters, AIC/BIC, Ljung-Box, skew and fat tails

Fitted once on all of 1928–2026, so **in-sample and for description only**: none of it enters the forecasts.

- **Persistence** = α + β + γ/2: how much of today's variance shock is still there tomorrow. The **half-life**, ln 0.5 / ln(persistence), is the number of days for a shock to halve; it is capped at 10 years because it explodes as persistence nears 1. The **long-run volatility** √(ω / (1 − persistence)) is where the forecast reverts to; it is left blank when persistence ≥ 0.999, where it is undefined or meaningless (the gaps in the charts of section 14).
- **AIC and BIC** compare models fitted to the same data: −2 × log-likelihood plus a penalty per parameter (2 for AIC, ln(n) for BIC, which penalises more). **Lower is better.** They say which model describes the data best, not which forecasts best.
- **Ljung-Box** tests whether a series is still autocorrelated. On the squared returns it shows volatility clustering; on the squared standardised residuals z² = (ε/σ)², a high p-value means the model has absorbed the clustering.
- **Skewness and excess kurtosis** of z: what is left of the asymmetry and fat tails after the model. A normal has 0 and 0; Jarque-Bera tests both at once.

In [ ]:
MAX_HALF_LIFE = 10 * 252  # cap, so near-integrated windows still give a usable number


def describe(res, mu_pct):
    p = res.params
    alpha, beta, gamma = p.get("alpha[1]", 0.0), p.get("beta[1]", 0.0), p.get("gamma[1]", 0.0)
    persistence = alpha + beta + gamma / 2
    return {
        "mu (annual)": mu_pct / 100 * 252,
        "omega": p["omega"],
        "alpha": alpha,
        "gamma": gamma if "gamma[1]" in p else np.nan,
        "beta": beta,
        "nu": p.get("nu", p.get("eta", np.nan)),  # degrees of freedom (eta in the skewed t)
        "lambda": p.get("lambda", np.nan),          # skew of the skewed t (negative = longer left tail)
        "persistence": persistence,
        "half-life (days)": min(np.log(0.5) / np.log(persistence), MAX_HALF_LIFE) if persistence < 1 else MAX_HALF_LIFE,
        "long-run vol (annual)": np.sqrt(p["omega"] / (1 - persistence) * 252) / 100 if persistence < 0.999 else np.nan,
    }

y_mean = y.mean()
full = {name: arch_model(y - y_mean, mean="Zero", **spec).fit(disp="off") for name, spec in MODELS.items()}
table = pd.DataFrame({name: describe(res, y_mean) for name, res in full.items()}).T
table["log-likelihood"] = [res.loglikelihood for res in full.values()]
table["parameters"] = [len(res.params) for res in full.values()]
table["AIC"] = [res.aic for res in full.values()]
table["BIC"] = [res.bic for res in full.values()]
table.round(4)

In [ ]:
def diagnostics(series, label):
    z = series.dropna()
    lb = acorr_ljungbox(z, lags=[10, 20], return_df=True)["lb_pvalue"]
    lb2 = acorr_ljungbox(z ** 2, lags=[10, 20], return_df=True)["lb_pvalue"]
    return {
        "series": label,
        "LB p, level, 10 lags": lb.loc[10],
        "LB p, level, 20 lags": lb.loc[20],
        "LB p, squared, 10 lags": lb2.loc[10],
        "LB p, squared, 20 lags": lb2.loc[20],
        "skewness": stats.skew(z),
        "excess kurtosis": stats.kurtosis(z),
        "Jarque-Bera p": stats.jarque_bera(z).pvalue,
    }

rows = [diagnostics((y - y.mean()) / y.std(), "raw returns (standardised)")]
rows += [diagnostics(res.std_resid, f"{name}: z = e/sigma") for name, res in full.items()]
pd.DataFrame(rows).set_index("series").round(4)

Reading the diagnostics: p-values near 0 on the *squared raw returns* confirm volatility clustering; the higher they are on the models' *squared standardised residuals*, the more of it the model has absorbed. Skewness and kurtosis that stay away from 0 after a model are what its shock distribution has to carry. With 25,000 daily observations even tiny departures are 'significant', so the size of the numbers matters more than the p-values. The returns themselves stay autocorrelated (Ljung-Box on the levels): that comes mostly from the early index, whose stale prices make daily returns look correlated, and affects the mean, not the variance.

## 12. Walk-forward: fit every month, forecast the next quarter

For each month-end from January 1938, take the 12 years of daily returns up to the trading day before the last and fit the four models. Each fit starts from the previous month's parameters, which makes it faster and steadier; if the optimiser fails to converge it is refitted from scratch, and if that fails too, last month's parameters are kept and the month is flagged.

The daily VaR for the days until the next refit is filtered through the actual returns with the month's parameters: for day t+1, σ²ₜ₊₁ = ω + (α + γ·[εₜ < 0])·ε²ₜ + β·σ²ₜ, and VaR = −(μ + σₜ₊₁·q), with q the 1% or 5% quantile of the model's shock distribution (for FHS, of the window's standardised residuals). Each day's VaR therefore uses only information up to the day before.

In [ ]:
def dist_quantiles(res, levels):
    dist = res.model.distribution
    names = dist.parameter_names()
    params = res.params[names].to_numpy() if names else None
    return np.asarray(dist.ppf(np.asarray(levels), params))


def fit_model(window, spec, start, prev_params):
    mu_pct = window.mean()  # the window's own mean: real time, nothing from the future
    am = arch_model(window - mu_pct, mean="Zero", **spec)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        res = am.fit(disp="off", starting_values=start, show_warning=False)
        status = "ok"
        if res.convergence_flag != 0:
            res = am.fit(disp="off", show_warning=False)
            status = "refit"
        if res.convergence_flag != 0 and prev_params is not None:
            res = am.fix(prev_params)
            status = "previous parameters"
    return res, mu_pct, status


def path_stats(paths, prefix):
    cum = paths.cumsum(axis=1)
    return {
        f"{prefix}p_sim_end": (cum[:, -1] <= np.log(0.9)).mean(),
        f"{prefix}p_sim_touch": (cum.min(axis=1) <= np.log(0.9)).mean(),
        f"{prefix}var95_3m": -np.expm1(np.quantile(cum[:, -1], 0.05)),
    }


def daily_var(res, mu_pct, sigma2_first, future, quantiles):
    # Filter the variance through the actual returns after the fit, one day at a time.
    p = res.params
    omega, alpha = p["omega"], p.get("alpha[1]", 0.0)
    gamma, beta = p.get("gamma[1]", 0.0), p.get("beta[1]", 0.0)
    out = np.empty((len(future), len(quantiles)))
    s2 = sigma2_first
    for i, r in enumerate(future.to_numpy()):
        out[i] = -(mu_pct + np.sqrt(s2) * quantiles) / 100
        e = r - mu_pct
        s2 = omega + (alpha + gamma * (e < 0)) * e * e + beta * s2
    return out


def walk_forward(months):
    rows, daily_rows = [], []
    prev = {name: None for name in MODELS}
    month_list = list(day_before_last.index)
    t0 = time.time()
    for i, m in enumerate(months):
        info = day_before_last[m]
        pos = y.index.get_loc(info)
        window = y.iloc[max(0, pos + 1 - WINDOW_DAYS): pos + 1]
        nxt = month_list.index(m) + 1
        end_pos = y.index.get_loc(day_before_last[month_list[nxt]]) if nxt < len(month_list) else len(y) - 1
        future = y.iloc[pos + 1: end_pos + 1]  # the days this month's fit is used for
        row = {"Date": m, "window_days": len(window), "window_vol": window.std() * np.sqrt(252) / 100}
        daily = pd.DataFrame(index=future.index)
        daily["ret"] = future / 100
        for name, spec in MODELS.items():
            res, mu_pct, status = fit_model(window, spec, prev[name], prev[name])
            prev[name] = res.params
            out = describe(res, mu_pct)
            out["status"] = status
            out["loglik"] = res.loglikelihood
            k, n = len(res.params), len(window)
            out["aic"] = -2 * res.loglikelihood + 2 * k
            out["bic"] = -2 * res.loglikelihood + np.log(n) * k
            tvalues = getattr(res, "tvalues", None)
            if tvalues is not None:
                out.update({f"t_{key.split('[')[0]}": v for key, v in tvalues.items()})

            mu_d = mu_pct / 100
            var_path = res.forecast(horizon=HORIZON, reindex=False).variance.values[-1]  # percent squared
            var_q = var_path[1:].sum() / 1e4
            out["vol_next_day"] = np.sqrt(var_path[0] * 252) / 100
            out["vol_q"] = np.sqrt(var_q / 0.25)
            out["p_closed"] = stats.norm.cdf((np.log(0.9) - 63 * mu_d) / np.sqrt(var_q))
            sims = res.forecast(horizon=HORIZON, method="simulation", simulations=N_SIMS, reindex=False,
                                random_state=np.random.RandomState(i))
            out.update(path_stats((sims.simulations.values[-1][:, 1:] + mu_pct) / 100, ""))
            row.update({f"{name}_{key}": v for key, v in out.items()})

            q = dist_quantiles(res, VAR_LEVELS)
            v = daily_var(res, mu_pct, var_path[0], future, q)
            for j, lvl in enumerate(VAR_LEVELS):
                daily[f"{name}_var{int(lvl * 100)}"] = v[:, j]

            if name == "gjr":
                z = res.std_resid.dropna()
                last = z.iloc[-21:]
                row["gjr_z_min_21"] = last.min()
                row["gjr_z_sum_21"] = last.sum()
                row["gjr_z_big_21"] = int((last.abs() > 2.5).sum())
                boot = res.forecast(horizon=HORIZON, method="bootstrap", simulations=N_SIMS, reindex=False,
                                    random_state=np.random.RandomState(10_000 + i))
                row.update(path_stats((boot.simulations.values[-1][:, 1:] + mu_pct) / 100, "gjr_fhs_"))
                v = daily_var(res, mu_pct, var_path[0], future, np.quantile(z, VAR_LEVELS))
                for j, lvl in enumerate(VAR_LEVELS):
                    daily[f"gjr_fhs_var{int(lvl * 100)}"] = v[:, j]
        rows.append(row)
        daily_rows.append(daily)
        if (i + 1) % 120 == 0:
            print(f"  {i + 1}/{len(months)} months, {time.time() - t0:.0f}s")
    print(f"{len(months)} months in {time.time() - t0:.1f}s")
    return pd.DataFrame(rows).set_index("Date"), pd.concat(daily_rows)


all_months = [m for m in day_before_last.index if FIRST_FORECAST <= m <= LAST_COMPLETE]
if GARCH_MODE == "test":
    picks = np.linspace(0, len(all_months) - 1, TEST_MONTHS).round().astype(int)
    garch, dvar = walk_forward([all_months[i] for i in picks])
elif RESULTS_FILE.exists() and DAILY_VAR_FILE.exists() and not GARCH_FORCE:
    garch = pd.read_csv(RESULTS_FILE, index_col="Date")
    garch.index = pd.PeriodIndex(garch.index, freq="M")
    dvar = pd.read_csv(DAILY_VAR_FILE, index_col=0, parse_dates=True)
    print(f"Read {len(garch)} months and {len(dvar)} days of saved results")
else:
    garch, dvar = walk_forward(all_months)
    garch.to_csv(RESULTS_FILE, float_format="%.6g")
    dvar.to_csv(DAILY_VAR_FILE, float_format="%.6g", date_format="%Y-%m-%d")
    print(f"Saved {RESULTS_FILE.relative_to(ROOT)} and {DAILY_VAR_FILE.relative_to(ROOT)}")

status_cols = [c for c in garch.columns if c.endswith("_status")]
print(garch[status_cols].apply(pd.Series.value_counts).fillna(0).astype(int))

## 13. How the forecasts do

On the same month-ends for every column: the GARCH family against trailing volatility, EWMA and the past base rate from section 9.

**Does GARCH predict volatility, or only repeat last month?** The forecast 3-month volatility against the volatility that then happened, the realised variance of the 63 trading days after the month-end. QLIKE is the standard loss for comparing variance forecasts (lower is better); it punishes forecasts that are too low more than ones that are too high. If GARCH only repeated the past, it would do no better than trailing volatility.

In [ ]:
sq = log_tr ** 2
month_end_day = log_tr.groupby(log_tr.index.to_period("M")).apply(lambda s: s.index[-1])
cum_sq = sq.cumsum()
realised = {}
for m, d in month_end_day.items():
    pos = log_tr.index.get_loc(d)
    if pos + 63 < len(log_tr):
        realised[m] = cum_sq.iloc[pos + 63] - cum_sq.iloc[pos]
realised = pd.Series(realised, name="rv_q")

ev = pd.DataFrame({"rv_q": realised})
ev["trailing63"] = (monthly["vol_trailing63"] ** 2) * 0.25
ev["ewma94"] = (monthly["vol_ewma94"] ** 2) * 0.25
for name in MODELS:
    ev[name] = (garch[f"{name}_vol_q"] ** 2) * 0.25
ev = ev.dropna()

def qlike(f, rv):
    return (rv / f - np.log(rv / f) - 1).mean()

vol_rows = {}
for col in ["trailing63", "ewma94", *MODELS]:
    vol_rows[col] = {
        "corr(log forecast, log realised)": np.corrcoef(np.log(ev[col]), np.log(ev["rv_q"]))[0, 1],
        "MSE of vol (annual)": ((np.sqrt(ev[col] / 0.25) - np.sqrt(ev["rv_q"] / 0.25)) ** 2).mean(),
        "QLIKE": qlike(ev[col], ev["rv_q"]),
    }
print(f"{ev.index.min()} to {ev.index.max()}: {len(ev)} month-ends")
pd.DataFrame(vol_rows).T.round(4)

**Does it predict 10% losses?** The same table as section 9, for every probability column.

In [ ]:
prob = pd.DataFrame({
    "event": monthly["event_tr"],
    "p_trailing63": monthly["p_trailing63"],
    "p_ewma94": monthly["p_ewma94"],
})
for name in MODELS:
    prob[f"p_{name}_closed"] = garch[f"{name}_p_closed"]
for name in SIM_VARIANTS:
    prob[f"p_{name}_sim_end"] = garch[f"{name}_p_sim_end"]
prob["p_base"] = monthly["event_tr"].shift(3).expanding(min_periods=12).mean()
prob = prob.dropna()

def score_table(frame, cols, event="event"):
    rows = {}
    base = brier(frame["p_base"], frame[event]) if "p_base" in frame else np.nan
    for col in cols:
        rows[col] = {
            "mean p before a loss": frame.loc[frame[event] == 1, col].mean(),
            "mean p otherwise": frame.loc[frame[event] == 0, col].mean(),
            "Brier": brier(frame[col], frame[event]),
            "Brier skill vs base rate": 1 - brier(frame[col], frame[event]) / base,
            "AUC": roc_auc_score(frame[event], frame[col]) if frame[event].nunique() > 1 else np.nan,
        }
    return pd.DataFrame(rows).T

print(f"{prob.index.min()} to {prob.index.max()}: {len(prob)} month-ends, {int(prob['event'].sum())} with a 10% loss")
score_table(prob, [c for c in prob.columns if c.startswith("p_")]).round(4)

**Losses that began in calm markets**: month-ends with a loss whose trailing volatility was below the median of months without one, where the normal model gives almost nothing (Olivia's finding, here on total return).

In [ ]:
calm_line = prob.loc[prob["event"] == 0].join(monthly["vol_trailing63"])["vol_trailing63"].median()
calm = prob.join(monthly["vol_trailing63"])
calm = calm[(calm["event"] == 1) & (calm["vol_trailing63"] < calm_line)]
print(f"Calm line: trailing volatility below {calm_line:.1%}; {len(calm)} loss months in calm markets")
calm[[c for c in calm.columns if c.startswith("p_")]].mean().sort_values().round(4)

## 14. The parameters over time

Exploratory: how each model's fitted parameters move as the 12-year window rolls, with the months followed by a 10% loss shaded.

Why they look the way they do:
- **GJR's α is near zero.** In GJR, α is the reaction to *rises* and α + γ the reaction to *falls*. For the S&P 500 rises barely raise volatility, so α sits at its lower bound of 0 and γ carries the response: the leverage effect.
- **β and persistence stay high**: daily volatility is very persistent, and a window that contains a change of regime (calm years, then a crisis) pushes the fitted persistence towards 1, because GARCH explains the shift as persistence (Lamoureux and Lastrapes, 1990).
- **The half-life explodes** because ln 0.5 / ln(persistence) magnifies tiny changes near 1: 0.98 gives 34 days, 0.995 gives 138, 0.999 gives 693. Persistence is the better series to use.
- **The long-run volatility has gaps** where persistence ≥ 0.999 and ω / (1 − persistence) is undefined or meaningless; they are left blank on purpose. The window's sample volatility is the stand-in used in section 17.

In [ ]:
param_plots = ["mu (annual)", "alpha", "gamma", "beta", "nu", "lambda", "persistence", "half-life (days)", "long-run vol (annual)", "vol_q"]
loss_months = prob.index[prob["event"] == 1] if len(prob) else []
fig, axes = plt.subplots(len(param_plots), 1, figsize=(12, 2.4 * len(param_plots)), sharex=True)
for ax, p in zip(axes, param_plots):
    for name in MODELS:
        col = f"{name}_{p}"
        if col in garch and garch[col].notna().any():
            s = garch[col].copy()
            s.index = s.index.to_timestamp()
            if p == "half-life (days)":
                s = s.clip(upper=500)
            ax.plot(s.index, s.values, lw=0.9, marker="." if GARCH_MODE == "test" else None, label=name, color=COLORS[name])
    for m in loss_months:
        ax.axvspan(m.to_timestamp(), (m + 1).to_timestamp(), color="red", alpha=0.08, lw=0)
    ax.set_ylabel(p, fontsize=8)
axes[0].legend(loc="upper left", ncol=4)
axes[0].set_title("Rolling 12-year GARCH parameters (red: month-ends followed by a 10% loss; half-life clipped at 500 days)")
plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
for name in ["garch", "gjr", "gjr_skewt", "gjr_fhs"]:
    s = garch[f"{name}_p_sim_end"].copy(); s.index = s.index.to_timestamp()
    ax[0].plot(s.index, s.values, lw=0.8, label=f"{name}, simulation", color=COLORS[name])
s = monthly["p_trailing63"].loc[garch.index.min():].copy(); s.index = s.index.to_timestamp()
ax[0].plot(s.index, s.values, lw=0.8, color="grey", label="trailing 63-day, normal")
for m in loss_months:
    ax[0].axvspan(m.to_timestamp(), (m + 1).to_timestamp(), color="red", alpha=0.08, lw=0)
ax[0].set_title("Probability of a 10% loss by the end of the quarter")
ax[0].legend(ncol=3, fontsize=8)
f = garch["gjr_vol_q"].copy(); f.index = f.index.to_timestamp()
r = np.sqrt(realised / 0.25).loc[garch.index.min():].copy(); r.index = r.index.to_timestamp()
ax[1].plot(r.index, r.values, lw=0.7, color="black", label="realised next 63 days")
ax[1].plot(f.index, f.values, lw=0.9, label="GJR forecast", color=COLORS["gjr"])
ax[1].set_title("GJR 3-month volatility forecast against what happened (annualized)")
ax[1].legend()
plt.tight_layout()

## 15. Model performance: likelihood ratio tests, significance, QQ plots

- **Likelihood ratio test**: LR = 2 × (log-likelihood of the bigger model − the smaller), compared with a χ² with as many degrees of freedom as the extra parameters, *when the smaller model sits inside the bigger one and the restriction is not on a boundary*.
  - t-GARCH → GJR-t: γ = 0 is an interior restriction, so χ²(1) is valid. This is the **test of the leverage effect**.
  - GJR-t → GJR-skew-t: λ = 0 gives back the t, so χ²(1).
  - GARCH → t-GARCH: the normal is the t with ν = ∞, a boundary, so the χ² p-value is not exact; the LR statistic and the AIC are reported instead.
- **Significance**: robust (Bollerslev–Wooldridge) t-statistics, the `arch` default. Each window's γ t-statistic is also tracked over time.
- **QQ plots**: the GJR residuals against the normal and against the fitted t; points off the line are the tails the distribution misses.

In [ ]:
pairs = [("garch", "tgarch", None), ("tgarch", "gjr", 1), ("gjr", "gjr_skewt", 1)]
lr_rows = []
for small, big, df in pairs:
    lr = 2 * (full[big].loglikelihood - full[small].loglikelihood)
    lr_rows.append({
        "test": f"{small} -> {big}",
        "LR": lr,
        "df": df if df else "boundary",
        "p-value": stats.chi2.sf(lr, df) if df else np.nan,
        "AIC drop": full[small].aic - full[big].aic,
        "BIC drop": full[small].bic - full[big].bic,
    })
display(pd.DataFrame(lr_rows).set_index("test").round(4))

sig = pd.concat({name: pd.DataFrame({"estimate": res.params, "robust t": res.tvalues, "p": res.pvalues})
                 for name, res in full.items()})
sig.round(4)

In [ ]:
# The same tests in every rolling window
lr_lev = 2 * (garch["gjr_loglik"] - garch["tgarch_loglik"])
lr_skew = 2 * (garch["gjr_skewt_loglik"] - garch["gjr_loglik"])
crit = stats.chi2.ppf(0.95, 1)
print(f"Windows where the leverage term is needed (LR > {crit:.2f}): {(lr_lev > crit).mean():.1%}")
print(f"Windows where gamma's robust t > 1.96: {(garch['gjr_t_gamma'] > 1.96).mean():.1%}")
print(f"Windows where the skew is needed (LR > {crit:.2f}): {(lr_skew > crit).mean():.1%}")

fig, ax = plt.subplots(3, 1, figsize=(12, 6.5), sharex=True)
for a, s, title in [(ax[0], lr_lev, "LR statistic, t-GARCH -> GJR (leverage)"),
                    (ax[1], garch["gjr_t_gamma"], "GJR gamma, robust t-statistic"),
                    (ax[2], lr_skew, "LR statistic, GJR-t -> GJR-skew-t")]:
    s = s.copy(); s.index = s.index.to_timestamp()
    a.plot(s.index, s.values, lw=0.9)
    a.axhline(crit if "LR" in title else 1.96, color="red", lw=0.8, ls="--")
    a.set_title(title, fontsize=9)
plt.tight_layout()

In [ ]:
z = full["gjr"].std_resid.dropna()
nu = full["gjr"].params["nu"]
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
stats.probplot(z, dist="norm", plot=ax[0])
ax[0].set_title("GJR-t residuals against the normal")
# The fitted t is standardised to variance 1, so scale the theoretical quantiles to match
osm = stats.t.ppf((np.arange(1, len(z) + 1) - 0.5) / len(z), nu) * np.sqrt((nu - 2) / nu)
ax[1].scatter(osm, np.sort(z), s=3)
lim = [min(osm.min(), z.min()), max(osm.max(), z.max())]
ax[1].plot(lim, lim, color="red", lw=0.8)
ax[1].set_title(f"GJR-t residuals against the fitted t (nu = {nu:.1f})")
ax[1].set_xlabel("theoretical quantiles"); ax[1].set_ylabel("ordered residuals")
plt.tight_layout()

## 16. Risk side: back-testing the VaR forecasts

Two VaR back-tests (Danielsson, chapter 8), then the same idea applied to our 10% probability.

- **Violation ratio (VR)** = violations observed / violations expected. Danielsson's rule of thumb: 0.8–1.2 good, below 0.5 or above 1.5 bad.
- **Bernoulli coverage test** (Kupiec): is the violation rate statistically equal to the VaR level? LR_uc ~ χ²(1).
- **Independence test** (Christoffersen): is a violation more likely the day after a violation? LR_ind ~ χ²(1). Clustering means the model reacts too slowly.
- **VaR volatility**: the standard deviation of the VaR forecast over time. With similar coverage, a steadier VaR is preferable.

**A. Daily VaR at 1% and 5%**, 1938–2026, about 22,000 days per model. **B. 3-month 95% VaR on non-overlapping quarters** (forecasts made in March, June, September and December only), which is closer to our question but has only about 350 observations; overlapping monthly windows would break the independence test.

In [ ]:
def bernoulli_test(v, p):
    n, x = len(v), int(v.sum())
    pi = x / n
    ll0 = (n - x) * np.log(1 - p) + x * np.log(p)
    ll1 = (n - x) * np.log(1 - pi) + x * np.log(pi) if 0 < x < n else 0.0
    lr = -2 * (ll0 - ll1)
    return lr, stats.chi2.sf(lr, 1)


def independence_test(v):
    v = np.asarray(v, dtype=int)
    a, b = v[:-1], v[1:]
    n00, n01 = int(((a == 0) & (b == 0)).sum()), int(((a == 0) & (b == 1)).sum())
    n10, n11 = int(((a == 1) & (b == 0)).sum()), int(((a == 1) & (b == 1)).sum())
    p01 = n01 / max(n00 + n01, 1)
    p11 = n11 / max(n10 + n11, 1)
    p = (n01 + n11) / max(n00 + n01 + n10 + n11, 1)
    def ll(prob, k0, k1):
        return (k0 * np.log(1 - prob) if k0 else 0) + (k1 * np.log(prob) if k1 else 0)
    lr = -2 * (ll(p, n00 + n10, n01 + n11) - ll(p01, n00, n01) - ll(p11, n10, n11))
    return lr, stats.chi2.sf(lr, 1)


def backtest(ret, var, p):
    ok = ret.notna() & var.notna()
    v = (ret[ok] < -var[ok]).astype(int)
    lr_uc, p_uc = bernoulli_test(v, p)
    lr_ind, p_ind = independence_test(v)
    return {"obs": len(v), "violations": int(v.sum()), "expected": len(v) * p,
            "VR": v.sum() / (len(v) * p), "Bernoulli p": p_uc, "independence p": p_ind,
            "VaR volatility": var[ok].std()}

rows = {}
for name in SIM_VARIANTS:
    for lvl in VAR_LEVELS:
        rows[(name, f"{lvl:.0%}")] = backtest(dvar["ret"], dvar[f"{name}_var{int(lvl * 100)}"], lvl)
print(f"A. Daily VaR, {dvar.index.min():%Y-%m-%d} to {dvar.index.max():%Y-%m-%d}")
pd.DataFrame(rows).T.round(4)

In [ ]:
quarters = garch.index[garch.index.month.isin([3, 6, 9, 12])]
ret_q = monthly["ret_3m_tr"].reindex(quarters)
rows = {}
for name in SIM_VARIANTS:
    rows[name] = backtest(ret_q, garch.loc[quarters, f"{name}_var95_3m"], 0.05)
print(f"B. 3-month 95% VaR on non-overlapping quarters, {quarters.min()} to {quarters.max()}")
pd.DataFrame(rows).T.round(4)

**The same test for our 10% loss probability**: if the probabilities are right, the expected number of losses (the sum of the forecasts) matches the number that happened, overall and within each band of forecasts. This is the calibration check behind the hit rates in Kritzman's Efficacy table.

In [ ]:
cal_rows = {}
for col in [c for c in prob.columns if c.startswith("p_")]:
    cal_rows[col] = {"expected losses": prob[col].sum(), "actual losses": int(prob["event"].sum()),
                     "ratio actual / expected": prob["event"].sum() / prob[col].sum()}
display(pd.DataFrame(cal_rows).T.round(2))

bands = pd.cut(prob["p_gjr_sim_end"], [0, 0.02, 0.05, 0.10, 0.20, 1.0], include_lowest=True)
pd.DataFrame({
    "months": prob.groupby(bands, observed=False)["event"].size(),
    "mean forecast": prob.groupby(bands, observed=False)["p_gjr_sim_end"].mean(),
    "actual loss rate": prob.groupby(bands, observed=False)["event"].mean(),
}).round(3)

## 17. Derived features for the other methods, and whether they are useful

Candidates built from the GARCH output, each known at the forecast date:

| Feature | What it says |
|---|---|
| `p_gjr` | GJR-t simulated probability of a 10% loss |
| `vol_ratio` | GJR's volatility for the next day over the window's sample volatility: how stretched volatility is against the era's normal (the sample volatility stands in for the model's long-run volatility, which is undefined in near-integrated windows) |
| `persistence` | GJR's α + β + γ/2 (rather than the half-life, which explodes near 1) |
| `z_min_21`, `z_sum_21`, `z_big_21` | The last 21 days' standardised residuals: the largest fall relative to the model, their sum, and how many were beyond ±2.5. A surprise can come before volatility catches up |
| `vrp` | VIX² − GJR's 3-month variance forecast: a variance risk premium (Bollerslev, Tauchen and Zhou, 2009). VXO before 1990, VIX after, taken on the trading day before month-end; from 1986 only |
| `vix_ratio` | VIX / GJR's 3-month volatility |

The checks:
1. **On its own**: AUC of each feature against the 10% loss event. Below 0.5 means the feature is higher before *calm* quarters; what matters is the distance from 0.5.
2. **Overlap**: rank correlations with each other and with trailing volatility, to spot features that say the same thing.
3. **Added to trailing volatility in a logistic regression**, walk-forward: refitted each January on the months whose outcome was already known, scored out of sample. Does adding the feature beat trailing volatility alone?

In [ ]:
def daily_series(path, col):
    d = pd.read_csv(path)
    d["observation_date"] = pd.to_datetime(d["observation_date"])
    return pd.to_numeric(d.set_index("observation_date")[col], errors="coerce").dropna() / 100

vix = daily_series(ROOT / "data/volatility/VIXCLS_raw.csv", "VIXCLS")
vxo = daily_series(ROOT / "data/volatility/VXOCLS_raw.csv", "VXOCLS")
implied = pd.concat([vxo[vxo.index < "1990-01-02"], vix]).sort_index()
implied_at = implied.reindex(pd.DatetimeIndex(day_before_last.values), method="ffill")
implied_at.index = day_before_last.index

feat = pd.DataFrame(index=garch.index)
feat["p_gjr"] = garch["gjr_p_sim_end"]
feat["vol_ratio"] = garch["gjr_vol_next_day"] / garch["window_vol"]
feat["persistence"] = garch["gjr_persistence"]
feat["z_min_21"] = garch["gjr_z_min_21"]
feat["z_sum_21"] = garch["gjr_z_sum_21"]
feat["z_big_21"] = garch["gjr_z_big_21"]
feat["vrp"] = implied_at.reindex(feat.index) ** 2 - garch["gjr_vol_q"] ** 2
feat["vix_ratio"] = implied_at.reindex(feat.index) / garch["gjr_vol_q"]
feat["vol_trailing63"] = monthly["vol_trailing63"]
feat["event"] = monthly["event_tr"]
feat = feat.dropna(subset=["event"])

auc_rows = {}
for col in [c for c in feat.columns if c != "event"]:
    s = feat[[col, "event"]].dropna()
    auc_rows[col] = {"from": str(s.index.min()), "months": len(s), "losses": int(s["event"].sum()),
                     "AUC": roc_auc_score(s["event"], s[col])}
print("1. Each feature on its own")
display(pd.DataFrame(auc_rows).T)

print("2. Rank correlations (from 1986, where every feature exists)")
feat.drop(columns="event").loc["1986":].corr(method="spearman").round(2)

In [ ]:
from sklearn.linear_model import LogisticRegression


def walk_forward_logit(frame, cols, first_year):
    # Refit each January on months whose 3-month outcome was known by then; predict that year's months.
    preds = []
    for year in range(first_year, frame.index.max().year + 1):
        start = pd.Period(f"{year}-01", "M")
        train = frame.loc[: start - 4].dropna(subset=cols + ["event"])
        test = frame.loc[start: pd.Period(f"{year}-12", "M")].dropna(subset=cols + ["event"])
        if test.empty or train["event"].nunique() < 2:
            continue
        mean, std = train[cols].mean(), train[cols].std()
        model = LogisticRegression(C=1e6, max_iter=1000).fit((train[cols] - mean) / std, train["event"])
        preds.append(pd.Series(model.predict_proba((test[cols] - mean) / std)[:, 1], index=test.index))
    return pd.concat(preds) if preds else pd.Series(dtype=float)


def compare_logits(frame, extra, first_year, label):
    base_cols = ["vol_trailing63"]
    out = {}
    specs = {"trailing vol only": base_cols, **{f"+ {c}": base_cols + [c] for c in extra}}
    preds = {k: walk_forward_logit(frame, v, first_year) for k, v in specs.items()}
    common = pd.concat(preds, axis=1).dropna()
    if common.empty or frame.loc[common.index, "event"].nunique() < 2:
        print(f"3. {label}: not enough months to fit (test mode?)")
        return pd.DataFrame()
    y_true = frame.loc[common.index, "event"]
    base_rate = frame["event"].shift(3).expanding(min_periods=12).mean().reindex(common.index)
    for k in specs:
        out[k] = {"AUC": roc_auc_score(y_true, common[k]),
                  "Brier": brier(common[k], y_true),
                  "Brier skill vs base rate": 1 - brier(common[k], y_true) / brier(base_rate, y_true)}
    print(f"3. {label}: {common.index.min()} to {common.index.max()}, {len(common)} months, {int(y_true.sum())} losses")
    return pd.DataFrame(out).T.round(4)

display(compare_logits(feat, ["p_gjr", "vol_ratio", "persistence", "z_min_21", "z_sum_21", "z_big_21"], 1960,
                       "Long sample, forecasts from 1960"))
compare_logits(feat, ["vrp", "vix_ratio"], 1996, "Implied-volatility features, forecasts from 1996 (training from 1986)")

Reading section 17: a feature earns a place if it has an AUC away from 0.5 on its own, is not a near-copy of trailing volatility or of another feature, and improves the out-of-sample logistic regression. The choice of columns for the team is left for later. The implied-volatility features have a short sample, with few losses, so their numbers are the least reliable.

## 18. The GARCH parameters as input variables

Section 17 tested persistence; here the same checks go to the other parameters of the rolling 12-year fits:

| Variable | What it measures |
|---|---|
| `gamma` | GJR's leverage term: how much more a fall raises volatility than a rise |
| `alpha_plus_gamma` | GJR's total reaction to a fall |
| `nu` | GJR's degrees of freedom: lower = fatter tails |
| `lambda` | The skewed-t's skew: more negative = longer left tail |
| `window_vol` | The sample volatility of the 12-year window |
| `t_gamma` | How clearly the window shows a leverage effect (γ's robust t-statistic) |
| `persistence` | α + β + γ/2, for comparison |

**Two cautions, built into the checks.**
1. **They move slowly.** Each comes from a 12-year window that changes by one month at a time, so neighbouring months are nearly identical and there are far fewer independent observations than months. The AUCs therefore get a **block-bootstrap 90% interval** (resampling 36-month blocks), and the table shows each variable's month-to-month autocorrelation and a rough count of independent observations.
2. **A past crisis in the window can drive them.** A window containing 1987 or 2008 has fatter tails and a higher volatility, and crises cluster in time, so a parameter can look predictive only because the window holds an earlier crisis. The proxy for that is `past_losses`: the number of 10% loss months known by the forecast date within the last 12 years. Test 4 asks whether each parameter still matters once trailing volatility and `past_losses` are in the regression (in-sample logit, standard errors robust to the overlap and autocorrelation).

In [ ]:
import statsmodels.api as sm

par = pd.DataFrame(index=garch.index)
par["gamma"] = garch["gjr_gamma"]
par["alpha_plus_gamma"] = garch["gjr_alpha"] + garch["gjr_gamma"]
par["nu"] = garch["gjr_nu"]
par["lambda"] = garch["gjr_skewt_lambda"]
par["window_vol"] = garch["window_vol"]
par["t_gamma"] = garch["gjr_t_gamma"]
par["persistence"] = garch["gjr_persistence"]
PARAMS = list(par.columns)
par["past_losses"] = monthly["event_tr"].shift(3).rolling(144, min_periods=1).sum().reindex(par.index)
par["vol_trailing63"] = monthly["vol_trailing63"]
par["p_gjr"] = garch["gjr_p_sim_end"]
par["event"] = monthly["event_tr"]
par = par.dropna(subset=["event"])

ERAS = [("1938", "1959"), ("1960", "1989"), ("1990", "2026")]
rng = np.random.default_rng(0)


def block_bootstrap_auc(x, ev, block=36, reps=1000):
    n = len(x)
    starts = np.arange(n - block + 1)
    out = []
    for _ in range(reps):
        idx = np.concatenate([np.arange(s, s + block) for s in rng.choice(starts, n // block + 1)])[:n]
        if ev[idx].sum() in (0, n):
            continue
        out.append(roc_auc_score(ev[idx], x[idx]))
    return np.percentile(out, [5, 95])


rows = {}
for col in PARAMS + ["vol_trailing63", "p_gjr"]:
    s = par[[col, "event"]].dropna()
    lo, hi = block_bootstrap_auc(s[col].to_numpy(), s["event"].to_numpy())
    rho1 = s[col].autocorr(1)
    row = {"AUC": roc_auc_score(s["event"], s[col]), "90% low": lo, "90% high": hi}
    for a, b in ERAS:
        e = s.loc[a:b]
        row[f"AUC {a}-{b[-2:]}"] = roc_auc_score(e["event"], e[col]) if e["event"].nunique() > 1 else np.nan
    row["autocorr 1m"] = rho1
    row["autocorr 12m"] = s[col].autocorr(12)
    row["~independent obs"] = len(s) * (1 - rho1) / (1 + rho1)
    rows[col] = row
print("1. Each variable on its own, with block-bootstrap intervals and by era")
pd.DataFrame(rows).T.round(3)

In [ ]:
print("2. Rank correlations with trailing volatility, the GARCH probability and the past-crisis proxy")
par[PARAMS + ["vol_trailing63", "p_gjr", "past_losses"]].corr(method="spearman").loc[PARAMS, ["vol_trailing63", "p_gjr", "past_losses"] + PARAMS].round(2)

In [ ]:
print("3. Added to trailing volatility in the walk-forward logistic regression (forecasts from 1960)")
display(compare_logits(par, PARAMS, 1960, "Parameters"))

print("4. In-sample logit: does each parameter matter beyond trailing volatility and past crises?")
rows = {}
for col in PARAMS:
    s = par[[col, "vol_trailing63", "past_losses", "event"]].dropna()
    X = (s[[col, "vol_trailing63", "past_losses"]] - s[[col, "vol_trailing63", "past_losses"]].mean()) / s[[col, "vol_trailing63", "past_losses"]].std()
    fit = sm.Logit(s["event"], sm.add_constant(X)).fit(disp=0, cov_type="HAC", cov_kwds={"maxlags": 12})
    rows[col] = {"coef (per 1 sd)": fit.params[col], "robust z": fit.tvalues[col], "p": fit.pvalues[col],
                 "z of past_losses": fit.tvalues["past_losses"]}
pd.DataFrame(rows).T.round(3)

Reading section 18: a parameter is worth passing on if its bootstrap interval stays clear of 0.5, its AUC points the same way in the different eras, it is not a near-copy of trailing volatility or of `past_losses`, and it keeps a robust z beyond about 2 in test 4. The walk-forward test is the strictest and, as section 17 showed, hard for any variable because the relationship changes between eras.

## 19. Ranking the months before a loss

AUC asks: take one month that preceded a loss and one that did not; how often did the forecast rank the loss month higher? Two different things pull it down:

- **(a) No information**: within an era, the signal does not separate risky months from the rest. Only new information helps.
- **(b) Drifting levels**: within each era the signal separates the months well, but its level drifts from era to era (a "high" volatility in the 1960s is an ordinary one in the 2000s), so months from different decades are ranked on the wrong scale. Then expressing the signal **relative to its own recent history** helps.

The checks:
1. **AUC within each decade** against the AUC over the whole sample. If the within-decade AUCs are clearly higher, (b) is at work.
2. **Relative versions**: each signal's percentile within its previous 10 years (120 months, past data only).
3. **The top of the ranking**, which is what Kritzman's Efficacy table looks at: for the months with the highest forecasts (top 10% and 20%, using thresholds set on past data only), how often a loss followed (precision), what share of all losses they caught (recall), and the average 3-month return, against all other months. And the same for fixed probability thresholds (5%, 10%, 20%).

In [ ]:
rank = pd.DataFrame({
    "vol_trailing63": monthly["vol_trailing63"],
    "p_trailing63": monthly["p_trailing63"],
    "p_gjr": garch["gjr_p_sim_end"],
    "p_gjr_skewt": garch["gjr_skewt_p_sim_end"],
    "persistence": garch["gjr_persistence"],
    "event": monthly["event_tr"],
    "ret_3m": monthly["ret_3m_tr"],
}).loc[FIRST_FORECAST:].dropna(subset=["event"])
SIGNALS = ["vol_trailing63", "p_gjr", "p_gjr_skewt", "persistence"]


def rolling_percentile(s, window=120, min_periods=60):
    # Share of the previous `window` months (this one included) with a lower value: past data only.
    return s.rolling(window, min_periods=min_periods).apply(lambda w: (w[:-1] < w[-1]).mean(), raw=True)


for col in SIGNALS:
    rank[f"{col}_rel"] = rolling_percentile(rank[col])

rows = {}
for col in SIGNALS + [f"{c}_rel" for c in SIGNALS]:
    s = rank[[col, "event"]].dropna()
    decade = (s.index.year // 10) * 10
    within = [roc_auc_score(g["event"], g[col]) for _, g in s.groupby(decade) if g["event"].nunique() > 1]
    rows[col] = {"pooled AUC": roc_auc_score(s["event"], s[col]),
                 "mean within-decade AUC": np.mean(within),
                 "decades with losses": len(within)}
    for a, b in ERAS:
        e = s.loc[a:b]
        rows[col][f"AUC {a}-{b[-2:]}"] = roc_auc_score(e["event"], e[col]) if e["event"].nunique() > 1 else np.nan
print("1-2. Pooled against within-decade AUC, raw and relative to the past 10 years")
pd.DataFrame(rows).T.round(3)

In [ ]:
def top_of_ranking(frame, col, top_share):
    # Threshold = the past (1 - top_share) quantile of the signal, from months before the forecast only.
    thr = frame[col].expanding(min_periods=120).quantile(1 - top_share).shift(1)
    s = frame.assign(thr=thr, flag=frame[col] > thr).dropna(subset=[col, "thr"])
    flagged, other = s[s["flag"]], s[~s["flag"]]
    return {"months flagged": len(flagged), "losses": int(s["event"].sum()),
            "loss rate when flagged": flagged["event"].mean(), "loss rate otherwise": other["event"].mean(),
            "share of losses caught": flagged["event"].sum() / max(s["event"].sum(), 1),
            "mean 3m return flagged": flagged["ret_3m"].mean(), "mean 3m return otherwise": other["ret_3m"].mean()}

rows = {}
for col in SIGNALS + ["p_gjr_rel", "vol_trailing63_rel"]:
    for top in [0.10, 0.20]:
        rows[(col, f"top {top:.0%}")] = top_of_ranking(rank, col, top)
print("3a. The top of each ranking (thresholds from past data only)")
display(pd.DataFrame(rows).T.round(3))

rows = {}
for col in ["p_trailing63", "p_gjr", "p_gjr_skewt"]:
    for thr in [0.05, 0.10, 0.20]:
        f = rank[rank[col] > thr]
        o = rank[rank[col] <= thr]
        rows[(col, f"> {thr:.0%}")] = {"months flagged": len(f), "loss rate when flagged": f["event"].mean(),
                                      "false alarms": int((f["event"] == 0).sum()),
                                      "share of losses caught": f["event"].sum() / rank["event"].sum(),
                                      "mean 3m return flagged": f["ret_3m"].mean(), "mean 3m return otherwise": o["ret_3m"].mean()}
print("3b. Fixed probability thresholds, as in Kritzman's Efficacy table")
pd.DataFrame(rows).T.round(3)

## 20. GARCH-X: outside information in the variance equation

GJR-t with exogenous variables added to tomorrow's variance (Danielsson, chapter 2-a, slides 59–62):

σ²ₜ = ω + (α + γ·[εₜ₋₁ < 0])·ε²ₜ₋₁ + β·σ²ₜ₋₁ + δ′·xₜ₋₁,  δ ≥ 0

Each x is known the day before it is used. The candidates in the repository with daily data:
- **Implied variance**: VIX² (VXO before 1990), converted to a daily variance. The market's own view of coming volatility.
- **Credit spread**: Moody's Baa yield minus the 10-year Treasury yield, daily. Stress in credit often comes before stress in equities.

Four models on identical windows: GJR-t without x (the baseline, refitted here with the same code so the comparison is fair), with VIX², with the credit spread, and with both.

**Limits.** Both series start in 1986, so the windows start there too, grow to 12 years by 1998, and the forecasts start in **1996** (ten years of data): about 370 months and only about 20 loss months, so every result is uncertain. The `arch` package cannot put variables in the variance equation, so the likelihood is written here: Student-t errors, the variance recursion run with a linear filter (fast without a compiler), maximised with L-BFGS-B. For forecasts the model needs the future path of x: each x is assumed to revert to its window mean like an AR(1) estimated on the window, which avoids freezing a high VIX for three months.

**What it is for**: whether outside information raises the AUC, which section 19 says needs *new* information. The likelihood ratio test of δ = 0 sits on a boundary (δ ≥ 0), so its 5% critical value is 2.71 rather than 3.84 (half χ²(0), half χ²(1)).

In [ ]:
from scipy.optimize import minimize
from scipy.signal import lfilter
from scipy.special import gammaln

X_FIRST_FORECAST = pd.Period("1996-01", "M")
X_RESULTS_FILE = ROOT / "data" / "volatility" / "garch_x_monthly_michal.csv"
X_FORCE = os.environ.get("GARCH_FORCE", "0") == "1"

dbaa = daily_series(ROOT / "data/credit/DBAA_raw.csv", "DBAA") * 100
dgs10 = daily_series(ROOT / "data/rates/DGS10_raw.csv", "DGS10") * 100
exog = pd.DataFrame({
    "vix2": (implied * 100) ** 2 / 252,  # implied daily variance, percent squared, like y
    "credit": dbaa - dgs10,               # Baa minus 10-year Treasury, percentage points
}).reindex(y.index).ffill()
exog = exog.loc[exog.dropna().index.min():]
print(f"Exogenous series from {exog.index.min():%Y-%m-%d}; latest: {exog.iloc[-1].round(3).to_dict()}")

X_MODELS = {"gjr_base": [], "gjrx_vix": ["vix2"], "gjrx_credit": ["credit"], "gjrx_both": ["vix2", "credit"]}


def gjrx_sigma2(theta, e, x):
    omega, alpha, gamma, beta = theta[:4]
    delta = theta[4:4 + x.shape[1]]
    u = np.empty_like(e)
    u[0] = np.var(e)
    u[1:] = omega + (alpha + gamma * (e[:-1] < 0)) * e[:-1] ** 2 + (x[:-1] @ delta if len(delta) else 0.0)
    s2 = np.empty_like(e)
    s2[0] = u[0]
    s2[1:] = lfilter([1.0], [1.0, -beta], u[1:], zi=[beta * u[0]])[0]
    return s2


def gjrx_negll(theta, e, x):
    nu = theta[-1]
    s2 = gjrx_sigma2(theta, e, x)
    if not np.all(np.isfinite(s2)) or np.any(s2 <= 0):
        return 1e12
    const = gammaln((nu + 1) / 2) - gammaln(nu / 2) - 0.5 * np.log(np.pi * (nu - 2))
    ll = const - 0.5 * np.log(s2) - (nu + 1) / 2 * np.log1p(e ** 2 / (s2 * (nu - 2)))
    return -ll.sum()


def fit_gjrx(e, x, start):
    k = x.shape[1]
    bounds = [(1e-6, 5), (0, 1), (0, 1), (0, 0.9999)] + [(0, 5)] * k + [(2.1, 200)]
    if start is None:
        start = np.array([0.02, 0.03, 0.10, 0.88] + [0.01] * k + [6.0])
    res = minimize(gjrx_negll, start, args=(e, x), method="L-BFGS-B", bounds=bounds)
    return res.x, -res.fun, res.success


def gjrx_forecast(theta, e, x, mu_pct, xbar, phi, seed):
    k = x.shape[1]
    omega, alpha, gamma, beta = theta[:4]
    delta, nu = theta[4:4 + k], theta[-1]
    s2 = gjrx_sigma2(theta, e, x)
    s2_next = omega + (alpha + gamma * (e[-1] < 0)) * e[-1] ** 2 + beta * s2[-1] + (x[-1] @ delta if k else 0.0)
    # Expected x on each forecast day: AR(1) reversion to the window mean
    h = np.arange(1, HORIZON)
    x_path = xbar + (phi[None, :] ** h[:, None]) * (x[-1] - xbar) if k else np.zeros((HORIZON - 1, 0))
    x_term = x_path @ delta if k else np.zeros(HORIZON - 1)
    pers = alpha + gamma / 2 + beta
    var_path = np.empty(HORIZON)
    var_path[0] = s2_next
    for i in range(1, HORIZON):
        var_path[i] = omega + x_term[i - 1] + pers * var_path[i - 1]
    var_q = var_path[1:].sum() / 1e4
    out = {"vol_next_day": np.sqrt(s2_next * 252) / 100, "vol_q": np.sqrt(var_q / 0.25),
           "p_closed": stats.norm.cdf((np.log(0.9) - 63 * mu_pct / 100) / np.sqrt(var_q))}
    r = np.random.RandomState(seed)
    scale = np.sqrt((nu - 2) / nu)
    s2_sim = np.full(N_SIMS, s2_next)
    paths = np.empty((N_SIMS, HORIZON))
    for i in range(HORIZON):
        z = r.standard_t(nu, N_SIMS) * scale
        eps = np.sqrt(s2_sim) * z
        paths[:, i] = eps
        if i < HORIZON - 1:
            s2_sim = omega + (alpha + gamma * (eps < 0)) * eps ** 2 + beta * s2_sim + x_term[i]
    out.update(path_stats((paths[:, 1:] + mu_pct) / 100, ""))
    return out


def walk_forward_x(months):
    rows, prev = [], {name: None for name in X_MODELS}
    t0 = time.time()
    for i, m in enumerate(months):
        info = day_before_last[m]
        win = y.loc[exog.index.min(): info].iloc[-WINDOW_DAYS:]
        mu_pct = win.mean()
        e = (win - mu_pct).to_numpy()
        row = {"Date": m, "window_days": len(win)}
        for name, cols in X_MODELS.items():
            xw = exog.loc[win.index, cols].to_numpy() if cols else np.zeros((len(win), 0))
            theta, ll, ok = fit_gjrx(e, xw, prev[name])
            prev[name] = theta
            if cols:
                xbar = xw.mean(axis=0)
                phi = np.array([np.clip(np.corrcoef(xw[1:, j], xw[:-1, j])[0, 1], 0, 0.9999) for j in range(len(cols))])
            else:
                xbar, phi = np.zeros(0), np.zeros(0)
            out = gjrx_forecast(theta, e, xw, mu_pct, xbar, phi, seed=i)
            out.update({"loglik": ll, "converged": ok, "omega": theta[0], "alpha": theta[1], "gamma": theta[2],
                        "beta": theta[3], "nu": theta[-1]})
            for j, c in enumerate(cols):
                out[f"delta_{c}"] = theta[4 + j]
            row.update({f"{name}_{k}": v for k, v in out.items()})
        rows.append(row)
        if (i + 1) % 60 == 0:
            print(f"  {i + 1}/{len(months)} months, {time.time() - t0:.0f}s")
    print(f"{len(months)} months in {time.time() - t0:.1f}s")
    return pd.DataFrame(rows).set_index("Date")


x_months = [m for m in day_before_last.index if X_FIRST_FORECAST <= m <= LAST_COMPLETE]
if GARCH_MODE == "test":
    gx = walk_forward_x(x_months[:: max(1, len(x_months) // 12)])
elif X_RESULTS_FILE.exists() and not X_FORCE:
    gx = pd.read_csv(X_RESULTS_FILE, index_col="Date")
    gx.index = pd.PeriodIndex(gx.index, freq="M")
    print(f"Read {len(gx)} months from {X_RESULTS_FILE.relative_to(ROOT)}")
else:
    gx = walk_forward_x(x_months)
    gx.to_csv(X_RESULTS_FILE, float_format="%.6g")
    print(f"Saved {X_RESULTS_FILE.relative_to(ROOT)}")
print("Converged:", {name: f"{gx[f'{name}_converged'].astype(bool).mean():.0%}" for name in X_MODELS})

In [ ]:
crit_boundary = 2.71  # 5% critical value of the half chi-squared(0), half chi-squared(1) mixture
rows = {}
for name, cols in X_MODELS.items():
    if not cols:
        continue
    lr = 2 * (gx[f"{name}_loglik"] - gx["gjr_base_loglik"])
    rows[name] = {"mean LR vs base": lr.mean(), "windows with LR > 2.71": (lr > crit_boundary).mean(),
                  **{f"mean delta_{c}": gx[f"{name}_delta_{c}"].mean() for c in cols}}
print("Does the outside variable improve the fit? (likelihood ratio against the baseline, every window)")
display(pd.DataFrame(rows).T.round(3))

fig, ax = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
for name, c in [("gjrx_vix", "vix2"), ("gjrx_credit", "credit"), ("gjrx_both", "vix2"), ("gjrx_both", "credit")]:
    s = gx[f"{name}_delta_{c}"].copy(); s.index = s.index.to_timestamp()
    ax[0].plot(s.index, s.values, lw=0.9, label=f"{name}: delta on {c}")
ax[0].legend(fontsize=8); ax[0].set_title("Weight on the outside variables (delta), rolling window")
for name in ["gjrx_vix", "gjrx_credit", "gjrx_both"]:
    s = 2 * (gx[f"{name}_loglik"] - gx["gjr_base_loglik"]); s.index = s.index.to_timestamp()
    ax[1].plot(s.index, s.values, lw=0.9, label=name)
ax[1].axhline(crit_boundary, color="red", ls="--", lw=0.8)
ax[1].set_title("Likelihood ratio against the baseline (red: 5% critical value)")
ax[1].legend(fontsize=8)
plt.tight_layout()

In [ ]:
evx = pd.DataFrame({"rv_q": realised})
evx["trailing63"] = (monthly["vol_trailing63"] ** 2) * 0.25
evx["gjr (arch, section 12)"] = (garch["gjr_vol_q"] ** 2) * 0.25
for name in X_MODELS:
    evx[name] = (gx[f"{name}_vol_q"] ** 2) * 0.25
evx = evx.dropna()
print(f"Volatility forecasts, {evx.index.min()} to {evx.index.max()}: {len(evx)} month-ends")
display(pd.DataFrame({c: {"QLIKE": qlike(evx[c], evx["rv_q"]),
                          "corr(log)": np.corrcoef(np.log(evx[c]), np.log(evx["rv_q"]))[0, 1]}
                      for c in evx.columns if c != "rv_q"}).T.round(4))

px = pd.DataFrame({"event": monthly["event_tr"], "p_trailing63": monthly["p_trailing63"],
                   "p_gjr_arch_sim_end": garch["gjr_p_sim_end"]})
for name in X_MODELS:
    px[f"p_{name}_sim_end"] = gx[f"{name}_p_sim_end"]
    px[f"p_{name}_closed"] = gx[f"{name}_p_closed"]
px["p_base"] = monthly["event_tr"].shift(3).expanding(min_periods=12).mean()
px = px.dropna()
print(f"10% loss probabilities, {px.index.min()} to {px.index.max()}: {len(px)} month-ends, {int(px['event'].sum())} losses")
score_table(px, [c for c in px.columns if c.startswith("p_")]).round(4)

Reading section 20: GARCH-X earns its place if the likelihood ratio says the outside variable is needed in most windows *and* the AUC rises against the baseline GJR on the same months. A better fit or a better volatility forecast alone would not help the project, whose weak point is ranking the months before a loss. With about 20 loss months, differences of a few hundredths in AUC are within noise.

## 21. Exports for the team

Two files, both derived (no prices), one row per calendar month-end with `Date` written like `data/equity-index/sp500_events.csv`:

1. **`data/equity-index/sp500_tr_events.csv`**: the total return version of the team's target, from the joined series of section 8 (Yahoo price + Shiller dividends to 1988-01-04, S&P's own total return index after). `Return_1m_TR`: the month's total return, close to close. `Return_3m_TR`: from the month-end close to the close three month-ends later. `Event_A_TR`: 1 if that is −10% or worse. `Event_B_TR`: 1 if any daily close in those three months is 10% or more below the starting close. Same definitions as Samantha's price-only file, on total return.
2. **`data/volatility/garch_features_michal.csv`**: the candidate predictors from this notebook, each known at the forecast date (data up to `signal_date`, the month's second-to-last trading day). Chosen to keep as much as might be useful to the logistic regression, the neural network and RBP, which can weigh variables differently for different forecasts. The weak ones are kept but marked in `data/README.md`, so nobody mistakes them for tested predictors. GARCH-X, the half-life and the long-run volatility are left out (section 20; capped and gappy series).

In [ ]:
# 1. Total return events
tr_lv = (1 + tr_daily).cumprod()
me_day = tr_lv.groupby(tr_lv.index.to_period("M")).apply(lambda s: s.index[-1])
lv_me = tr_lv.reindex(me_day.values)
lv_me.index = me_day.index
ev_rows = []
months = list(me_day.index)
for i, m in enumerate(months):
    row = {"Date": m.to_timestamp(how="end").strftime("%Y-%m-%d"),
           "Return_1m_TR": lv_me.iloc[i] / lv_me.iloc[i - 1] - 1 if i > 0 else np.nan}
    if i + 3 < len(months):
        start, end = me_day.iloc[i], me_day.iloc[i + 3]
        path = tr_lv.loc[start:end].iloc[1:] / tr_lv.loc[start] - 1
        row["Return_3m_TR"] = lv_me.iloc[i + 3] / lv_me.iloc[i] - 1
        row["Event_A_TR"] = int(row["Return_3m_TR"] <= -0.10)
        row["Event_B_TR"] = int((path <= -0.10).any())
    ev_rows.append(row)
tr_events = pd.DataFrame(ev_rows).dropna(subset=["Return_3m_TR"])
tr_events[["Event_A_TR", "Event_B_TR"]] = tr_events[["Event_A_TR", "Event_B_TR"]].astype(int)
tr_events.to_csv(ROOT / "data/equity-index/sp500_tr_events.csv", index=False, float_format="%.6f")

old = pd.read_csv(ROOT / "data/equity-index/sp500_events.csv")
both_ev = old.merge(tr_events, on="Date")
print(f"sp500_tr_events.csv: {tr_events['Date'].iloc[0]} to {tr_events['Date'].iloc[-1]}, {len(tr_events)} month-ends")
print(f"Event A: total return {tr_events['Event_A_TR'].sum()}, price only {old['Event_A'].sum()}; "
      f"Event B: total return {tr_events['Event_B_TR'].sum()}, price only {old['Event_B'].sum()}")
print(f"Price-only 3m returns vs total return, same months: mean gap {(both_ev['Return_3m_TR'] - both_ev['Return_3m']).mean():.4f}")

In [ ]:
# 2. Candidate predictors
out = pd.DataFrame(index=monthly.index)
out["signal_date"] = pd.Series(day_before_last).reindex(out.index).dt.strftime("%Y-%m-%d")
# Trailing volatility and EWMA on total return (section 9)
out["vol_trailing63_tr"] = monthly["vol_trailing63"]
out["vol_ewma94_tr"] = monthly["vol_ewma94"]
out["mu_expanding_tr"] = monthly["mu_expanding"]
out["p_trailing63_tr"] = monthly["p_trailing63"]
out["p_ewma94_tr"] = monthly["p_ewma94"]
out["vol_trailing63_rel"] = rolling_percentile(monthly["vol_trailing63"])
# The two best GARCH models (sections 15-16): GJR-t and GJR-skew-t
for name in ["gjr", "gjr_skewt"]:
    for k in ["p_sim_end", "p_sim_touch", "p_closed", "vol_q", "vol_next_day", "var95_3m"]:
        out[f"{name}_{k}"] = garch[f"{name}_{k}"]
out["gjr_skewt_p_rel"] = rolling_percentile(garch["gjr_skewt_p_sim_end"]).reindex(out.index)
# Derived from the GJR fit (section 17)
out["gjr_vol_ratio"] = feat["vol_ratio"]
for k in ["z_min_21", "z_sum_21", "z_big_21"]:
    out[f"gjr_{k}"] = garch[f"gjr_{k}"]
out["vrp"] = feat["vrp"]
out["vix_ratio"] = feat["vix_ratio"]
# Parameters of the rolling 12-year fits (section 18): weak on their own, kept for the ML methods and RBP
out["gjr_gamma"] = garch["gjr_gamma"]
out["gjr_alpha_plus_gamma"] = garch["gjr_alpha"] + garch["gjr_gamma"]
out["gjr_nu"] = garch["gjr_nu"]
out["gjr_t_gamma"] = garch["gjr_t_gamma"]
out["gjr_persistence"] = garch["gjr_persistence"]
out["gjr_skewt_lambda"] = garch["gjr_skewt_lambda"]
out["garch_window_vol"] = garch["window_vol"]

out = out.loc[out.drop(columns="signal_date").notna().any(axis=1)]
out.index = out.index.to_timestamp(how="end").strftime("%Y-%m-%d")
out.index.name = "Date"
out.to_csv(ROOT / "data/volatility/garch_features_michal.csv", float_format="%.6g")
starts = {c: out[c].first_valid_index() for c in out.columns if c != "signal_date"}
print(f"garch_features_michal.csv: {out.index[0]} to {out.index[-1]}, {len(out)} month-ends, {out.shape[1] - 1} columns")
pd.Series(starts, name="first month").to_frame()